# Dynamic Phi Model Training Demo

This notebook demonstrates how to use the dynamic_phi.py script for training Phi models with GRPO (Generative Reward Policy Optimization). It shows how to configure the training process, prepare data, and analyze results.

In [ ]:
import os
import sys
import torch
import wandb
import logging
from datasets import load_dataset, Dataset
from datetime import datetime
import re

# Ensure the project root is in sys.path for imports
project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import necessary modules
from grpo.config import RewardConfig
from grpo.dynamic_reward import DynamicReward
from utils.similarity_checker import SolutionSimilarityChecker
from utils.data_preparationphi import prepare_combined_data

## Setup Logging

First, let's set up logging to track our progress.

In [ ]:
def setup_logging(model_type: str) -> logging.Logger:
    """Setup logging configuration"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_dir = f"logs/{model_type}"
    os.makedirs(log_dir, exist_ok=True)
    
    logger = logging.getLogger('dynamic_grpo')
    
    # Clear any existing handlers to prevent duplicate logging
    if logger.handlers:
        logger.handlers.clear()
        
    logger.setLevel(logging.INFO)
    
    file_handler = logging.FileHandler(
        f"{log_dir}/notebook_{timestamp}.log"
    )
    file_handler.setFormatter(logging.Formatter(
        '%(asctime)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
    logger.addHandler(file_handler)
    logger.addHandler(logging.StreamHandler())
    return logger

# Initialize logger
logger = setup_logging("phi_notebook")
logger.info("Notebook started")

## Configure Reward Function

Now let's set up the reward configuration and initialize the DynamicReward class.

In [ ]:
# Initialize config
reward_config = RewardConfig(model_type="phi_notebook")
reward_config.group_diversity_bonus = 2.0  # Increased from default

# Setup
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"notebook_results/{reward_config.model_type}/{timestamp}"
os.makedirs(output_dir, exist_ok=True)

# Initialize similarity checker
similarity_checker = SolutionSimilarityChecker(reward_config)

# Initialize dynamic reward function
reward_func = DynamicReward(reward_config, similarity_checker)
logger.info("Initialized DynamicReward")
logger.info(f"Has stats object: {hasattr(reward_func, 'stats')}")

# Print initial stats configuration
if hasattr(reward_func, 'stats'):
    logger.info("Initial stats configuration:")
    for category in ['reward_components', 'group_stats', 'step_stats', 'similarity_stats']:
        if hasattr(reward_func.stats, category):
            stats_dict = getattr(reward_func.stats, category)
            logger.info(f"{category}: {stats_dict}")

## Define System Prompts

Let's define the system prompts used for different example types.

In [ ]:
# System prompts
FULLSOLUTION_SYSTEM_PROMPT = """\
You are a helpful math tutor. Solve the given problem step-by-step, showing your work clearly.

First, think through the problem carefully in the <thinking> section. This is your private scratchpad.

Then, provide your formal solution in the <answer> section, using <step> tags for each step.

For the final answer, use \boxed{} notation.
"""

COMPLETION_SYSTEM_PROMPT = """\
You are a helpful math tutor. Continue the partial solution to complete the problem.

Follow the same format and style as the partial solution provided.
Continue with the next logical step and complete the solution.
Make sure to use \boxed{} notation for the final answer.
"""

PROGRAMMER_SYSTEM_PROMPT = """\
You are a helpful math tutor who solves problems using Python code.

First, think through the problem carefully in the <thinking> section. This is your private scratchpad.

Then, provide your solution in the <answer> section with the following structure:
1. A brief explanation of your approach
2. Python code that solves the problem
3. The final answer using \boxed{} notation
"""

# Print token counts (simulated)
print(f"Solver system prompt length: {len(FULLSOLUTION_SYSTEM_PROMPT)} characters")
print(f"Completion system prompt length: {len(COMPLETION_SYSTEM_PROMPT)} characters")
print(f"Programmer system prompt length: {len(PROGRAMMER_SYSTEM_PROMPT)} characters")

## Load and Prepare Dataset

Now let's load and prepare a small sample dataset for demonstration purposes.

In [ ]:
# Function to count tokens (simulated)
def count_tokens(text):
    # Simple approximation: 1 token ≈ 4 characters
    return len(text) // 4

def get_sample_questions(num_examples=10):
    """Create a small sample dataset for demonstration"""
    # Create a simple dataset with math problems
    problems = [
        "Solve for x: 2x + 3 = 7",
        "Find the derivative of f(x) = x^2 + 3x + 1",
        "Evaluate the integral of x^2 from 0 to 2",
        "If a triangle has sides of length 3, 4, and 5, what is its area?",
        "Solve the system of equations: x + y = 5, 2x - y = 1",
        "Find the roots of the quadratic equation: x^2 - 5x + 6 = 0",
        "Calculate the limit of (x^2 - 1)/(x - 1) as x approaches 1",
        "If f(x) = 3x^2 and g(x) = 2x + 1, find (f ∘ g)(2)",
        "Find the sum of the first 10 terms of the arithmetic sequence: 3, 7, 11, ...",
        "A ball is thrown upward with an initial velocity of 20 m/s. How high will it go?"
    ]
    
    answers = [
        "x = 2",
        "f'(x) = 2x + 3",
        "8/3",
        "6",
        "x = 2, y = 3",
        "x = 2, x = 3",
        "2",
        "75",
        "210",
        "20.4 meters"
    ]
    
    # Create dataset
    dataset_dict = {
        "problem": problems[:num_examples],
        "answer": answers[:num_examples]
    }
    
    return Dataset.from_dict(dataset_dict)

# Get sample dataset
sample_dataset = get_sample_questions()
print(f"Sample dataset size: {len(sample_dataset)} examples")
print("First example:")
print(sample_dataset[0])

In [ ]:
# Prepare the dataset with combined example types
distribution = {
    'solution': 0.35,
    'programming': 0.35,
    'completion': 0.15,
    'wait': 0.15
}

formatted_dataset = prepare_combined_data(
    sample_dataset, 
    FULLSOLUTION_SYSTEM_PROMPT, 
    COMPLETION_SYSTEM_PROMPT, 
    PROGRAMMER_SYSTEM_PROMPT,
    None,  # No tokenizer for this demo
    distribution
)

# Shuffle the dataset
formatted_dataset = formatted_dataset.shuffle(seed=42)

# Count example types
example_types = {}
for example in formatted_dataset:
    et = example.get('example_type', 'unknown')
    example_types[et] = example_types.get(et, 0) + 1

print(f"Example types in dataset: {example_types}")

## Examine Dataset Examples

Let's look at examples of each type in our dataset.

In [ ]:
# Function to find an example of a specific type
def find_example_by_type(dataset, example_type):
    for i, example in enumerate(dataset):
        if example.get('example_type') == example_type:
            return i, example
    return None, None

# Find examples of each type
example_types = ['solution', 'completion', 'programming', 'wait']
for et in example_types:
    idx, example = find_example_by_type(formatted_dataset, et)
    if example:
        print(f"\n{'-'*50}\nExample of type '{et}' (index {idx}):\n{'-'*50}")
        print(f"Problem: {example.get('problem', 'N/A')}")
        print(f"Answer: {example.get('answer', 'N/A')}")
        
        # Print prompt (truncated for readability)
        prompt = example.get('prompt', '')
        if len(prompt) > 200:
            prompt = prompt[:200] + "..."
        print(f"Prompt (truncated): {prompt}")
        
        # For completion examples, show partial solution
        if et == 'completion' and 'partial_solution' in example:
            partial = example.get('partial_solution', '')
            if len(partial) > 200:
                partial = partial[:200] + "..."
            print(f"Partial solution (truncated): {partial}")
            
        # Count tokens
        prompt_tokens = count_tokens(example.get('prompt', ''))
        print(f"Estimated prompt tokens: {prompt_tokens}")

## Simulate Training Process

Let's simulate the training process by processing a batch of examples through the reward function.

In [ ]:
# Function to simulate model completions
def simulate_completions(examples, num_generations=5):
    """Simulate model completions for a batch of examples"""
    all_completions = []
    
    for example in examples:
        example_type = example.get('example_type')
        
        # Generate different completions based on example type
        if example_type == 'solution':
            # Simulate solution completions
            completions = [
                f"<thinking>\nLet me solve this step by step.\n</thinking>\n\n<answer>\n<step>First step for solution {i}</step>\n<step>Second step for solution {i}</step>\n\nThe answer is \\boxed{{{example.get('answer')}}}.\n</answer>"
                for i in range(num_generations)
            ]
        elif example_type == 'completion':
            # Simulate completion continuations
            partial = example.get('partial_solution', '')
            completions = [
                f"<step>Next step for completion {i}</step>\n<step>Final step for completion {i}</step>\n\nThe answer is \\boxed{{{example.get('answer')}}}.\n</answer>"
                for i in range(num_generations)
            ]
        elif example_type == 'programming':
            # Simulate programming solutions
            completions = [
                f"<thinking>\nI'll solve this with Python.\n</thinking>\n\n<answer>\nHere's my approach:\n\n```python\n# Solution {i}\nresult = {example.get('answer')}\nprint(result)\n```\n\nThe answer is \\boxed{{{example.get('answer')}}}.\n</answer>"
                for i in range(num_generations)
            ]
        elif example_type == 'wait':
            # Simulate wait examples
            completions = [
                f"Wait a second, let me think about this more carefully.\n\n<thinking>\nAdditional thinking for wait example {i}\n</thinking>\n\n<answer>\n<step>First step for wait {i}</step>\n<step>Second step for wait {i}</step>\n\nThe answer is \\boxed{{{example.get('answer')}}}.\n</answer>"
                for i in range(num_generations)
            ]
        else:
            # Default completions
            completions = [
                f"Default completion {i} with answer {example.get('answer')}"
                for i in range(num_generations)
            ]
            
        all_completions.append(completions)
    
    return all_completions

# Select a small batch of examples
batch_size = 3
batch_examples = formatted_dataset.select(range(batch_size))

# Simulate completions
num_generations = 5
batch_completions = simulate_completions(batch_examples, num_generations)

print(f"Generated {len(batch_completions)} sets of completions with {num_generations} generations each")

In [ ]:
# Process the batch through the reward function
def process_batch(examples, completions):
    """Process a batch of examples through the reward function"""
    batch_rewards = []
    
    for i, example in enumerate(examples):
        # Extract example data
        prompt = example.get('prompt', '')
        answer = example.get('answer', '')
        example_type = example.get('example_type', 'unknown')
        partial_solution = example.get('partial_solution', None)
        
        # Prepare kwargs for reward function
        kwargs = {
            'prompts': [prompt] * num_generations,
            'answer': [answer] * num_generations,
            'example_type': [example_type] * num_generations
        }
        
        if partial_solution is not None:
            kwargs['partial_solution'] = [partial_solution] * num_generations
        
        # Calculate rewards
        rewards = reward_func(completions[i], **kwargs)
        batch_rewards.append(rewards)
        
        # Print results
        print(f"\nExample {i+1} ({example_type}):\nPrompt: {prompt[:50]}...")
        print(f"Answer: {answer}")
        print(f"Rewards: {[round(r, 2) for r in rewards]}")
        print(f"Average reward: {sum(rewards)/len(rewards):.2f}")
    
    return batch_rewards

# Process the batch
batch_rewards = process_batch(batch_examples, batch_completions)

## Analyze Reward Statistics

Let's examine the reward statistics collected during processing.

In [ ]:
# Print reward statistics
if hasattr(reward_func, 'stats'):
    print("\nReward Statistics Summary:")
    print(reward_func.stats.get_summary())
    
    print("\nReward Components:")
    for key, value in reward_func.stats.reward_components.items():
        print(f"  {key}: {value}")
    
    # Check for other stat categories
    for category in ['group_stats', 'step_stats', 'similarity_stats', 'programming_stats']:
        if hasattr(reward_func.stats, category):
            stats_dict = getattr(reward_func.stats, category)
            if stats_dict:
                print(f"\n{category.replace('_', ' ').title()}:")
                for key, value in stats_dict.items():
                    print(f"  {key}: {value}")
else:
    print("No statistics available.")

## Visualize Results

Let's visualize the rewards for different example types.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare data for visualization
example_types = [example.get('example_type') for example in batch_examples]
avg_rewards = [sum(rewards)/len(rewards) for rewards in batch_rewards]
max_rewards = [max(rewards) for rewards in batch_rewards]
min_rewards = [min(rewards) for rewards in batch_rewards]

# Create bar chart
plt.figure(figsize=(10, 6))
x = np.arange(len(example_types))
width = 0.25

plt.bar(x - width, avg_rewards, width, label='Average Reward', color='blue')
plt.bar(x, max_rewards, width, label='Max Reward', color='green')
plt.bar(x + width, min_rewards, width, label='Min Reward', color='red')

plt.xlabel('Example Type')
plt.ylabel('Reward Value')
plt.title('Rewards by Example Type')
plt.xticks(x, example_types)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated how to use the dynamic_phi.py script for training Phi models with GRPO. We've seen how to:

1. Configure the reward function
2. Prepare a dataset with different example types
3. Process examples through the reward function
4. Analyze reward statistics
5. Visualize the results

For actual training, you would use the full dynamic_phi.py script, which includes model loading, GRPO training configuration, and model saving.